# CXR training on Colab (T4)

Mounts Google Drive for checkpoint persistence (Colab sessions disconnect; this is how you survive that), installs deps, downloads NIH ChestX-ray14, and runs `train.train`.

Runtime > Change runtime type > GPU (T4) before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/multimodal-medical-imaging-assistant/checkpoints'
!mkdir -p "$CHECKPOINT_DIR"

In [ ]:
# Replace with your repo URL once pushed to GitHub.
!git clone https://github.com/<you>/multimodal-medical-imaging-assistant.git
%cd multimodal-medical-imaging-assistant
!pip install -q -r requirements.txt

In [ ]:
# Kaggle API token: upload kaggle.json (from kaggle.com/settings) via the Colab file browser first.
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!python -m data.download nih --dest data_raw/nih_cxr
!python -m data.prepare_nih --root data_raw/nih_cxr

In [ ]:
import wandb
wandb.login()  # skip if you'd rather leave train.wandb_mode=disabled

In [ ]:
!python -m train.train --modality cxr \
  --set train.checkpoint_dir=$CHECKPOINT_DIR \
  --set train.wandb_mode=online \
  --set data.cxr.root=data_raw/nih_cxr \
  --set data.cxr.labels_csv=data_raw/nih_cxr/labels.csv

If the runtime disconnects, just re-run the training cell -- `train.train` auto-resumes from the latest checkpoint in `CHECKPOINT_DIR` on Drive.